# Demo multiagente -- Agente de produccion de reels 3D

Avance inicial para la materia de Agentes Inteligentes. Este notebook corre
un sistema de **6 agentes especializados** (Orquestador, Investigacion,
Guion, Voz, Visual/3D y QA) que colaboran, bajo un estado compartido
(`ReelState`), para producir la especificacion completa de un reel corto
-- sin generar el video real.

**Alcance de esta demo** (ver README.md para el detalle):
- Los agentes de Investigacion, Voz y Visual usan **herramientas reales
  del Agent SDK** (`@tool`), pero esas herramientas estan **mockeadas**:
  no llaman a OmniVoice ni a Remotion, simulan su resultado.
- Las llamadas al modelo **si son reales** -- cada agente invoca a Claude
  a traves del Agent SDK, usando la sesion de suscripcion de Claude Code
  (nunca una API key; ver `agents/_sdk_helpers.py`).
- El agente de QA es deliberadamente **codigo Python puro**, no un LLM
  (ver `agents/qa_agent.py` para la justificacion).


In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from state import ReelState
from agents._sdk_helpers import ejecutar

print("Repo root:", ROOT)


Repo root: C:\Users\aleja\Documents\Projects\agentes-reels


## 1. Perfilamiento y memoria compartida

Cada agente tiene un rol fijo, escrito en `prompts/*.txt` (perfilamiento,
ver ARCHITECTURE.md). La memoria del sistema es un solo objeto, `ReelState`,
que cada agente lee y actualiza -- no hay memoria entre corridas del
notebook, solo dentro de esta ejecucion.

El `brief` de abajo es el unico dato que inventamos nosotros (el tema y la
duracion deseada); todo lo demas lo produce el sistema de agentes.

In [2]:
brief = {
    "tema": "ahorro_voluntario",
    "cliente_demo": "Cooperativa Demo (dato ficticio para esta demo)",
    "duracion_objetivo_s": 20,
}
state = ReelState(brief=brief)
state


ReelState(brief={'tema': 'ahorro_voluntario', 'cliente_demo': 'Cooperativa Demo (dato ficticio para esta demo)', 'duracion_objetivo_s': 20}, plan=[], dossier={}, guion=[], voz=[], visual=[], qa={}, intentos_replanificacion=0, log=[])

## 2. Planificacion -- el Orquestador decide el plan de fases

In [3]:
from agents import orchestrator

plan_resultado = ejecutar(orchestrator.planificar(state))
plan_resultado


{'plan': {'fases': ['investigacion', 'guion', 'voz_y_visual', 'qa'],
  'razon': 'El tema ahorro_voluntario exige fundamentos verificables antes de redactar, así que la investigación va primero y debe limitarse a conceptos generales, marcando como pendiente cualquier cifra o condición de producto, ya que el cliente es ficticio. El guion depende de esos hallazgos y debe ajustarse a unos 20 segundos. Voz y visual se ejecutan juntos porque ambos derivan del mismo guion aprobado y deben sincronizarse en tiempo. QA cierra el flujo para verificar duración, formato vertical 1080x1920 y ausencia de datos de negocio inventados; si falla, el orquestador debe replanificar desde la fase afectada.'},
 'uso': {'input_tokens': 2,
  'cache_creation_input_tokens': 7163,
  'cache_read_input_tokens': 0,
  'output_tokens': 732,
  'output_tokens_details': {'thinking_tokens': 229},
  'server_tool_use': {'web_search_requests': 0, 'web_fetch_requests': 0},
  'service_tier': 'standard',
  'cache_creation': {'ep

## 3. Accion -- Investigacion

Este agente solo puede llamar a una herramienta: `leer_fuente`. Es una
fuente de datos simulada (en el motor real seria leer codigo o documentos
del cliente) -- si el tema no esta ahi, el agente debe decirlo, no
inventar.

In [4]:
from agents import research_agent

inv_resultado = ejecutar(research_agent.investigar(state, tema=brief["tema"]))
inv_resultado


{'dossier': {'tema': 'ahorro_voluntario',
  'datos': {'producto': 'Ahorro voluntario programado',
   'beneficio_clave': 'los aportes quedan disponibles en cualquier momento, sin plazo fijo',
   'publico': 'asociados activos de la cooperativa',
   'dato_ejemplo': 'tasa de referencia DTF + 2 puntos'},
  'advertencias': ['El dato de ejemplo (DTF + 2 puntos) es una cifra de catálogo demo según la fuente; debe confirmarse con el cliente antes de publicarse.',
   'La fuente no incluye tasa efectiva verificada, condiciones de retiro, montos mínimos ni plazos; no se reportan.']},
 'uso': {'input_tokens': 4,
  'cache_creation_input_tokens': 11159,
  'cache_read_input_tokens': 7526,
  'output_tokens': 740,
  'output_tokens_details': {'thinking_tokens': 224},
  'server_tool_use': {'web_search_requests': 0, 'web_fetch_requests': 0},
  'service_tier': 'standard',
  'cache_creation': {'ephemeral_1h_input_tokens': 11159,
   'ephemeral_5m_input_tokens': 0},
  'inference_geo': 'global',
  'iterations':

## 4. Guion -- convierte el dossier verificado en una tabla de escenas

In [5]:
from agents import scriptwriter_agent

guion_resultado = ejecutar(scriptwriter_agent.escribir_guion(
    state, duracion_objetivo_s=brief["duracion_objetivo_s"]
))
guion_resultado


{'guion': {'escenas': [{'id': 'E1_gancho',
    'duracion_s': 3,
    'texto_pantalla': '¿Tu ahorro, atado a un plazo?',
    'voz': '¿Y si tu ahorro no estuviera atado a un plazo fijo?',
    'visual_idea': 'Plano cerrado de un calendario con una cadena que se rompe; transición rápida a fondo limpio con colores de la cooperativa.'},
   {'id': 'E2_producto',
    'duracion_s': 3,
    'texto_pantalla': 'Ahorro voluntario programado',
    'voz': 'Conoce el ahorro voluntario programado de tu cooperativa.',
    'visual_idea': 'Logo de la cooperativa y el nombre del producto aparecen con animación de texto sobre fondo en movimiento suave.'},
   {'id': 'E3_beneficio_1',
    'duracion_s': 4,
    'texto_pantalla': 'Tus aportes, disponibles en cualquier momento',
    'voz': 'Tus aportes quedan disponibles en cualquier momento.',
    'visual_idea': 'Persona consultando su celular con gesto de tranquilidad; ícono de candado abierto superpuesto.'},
   {'id': 'E4_beneficio_2',
    'duracion_s': 4,
    '

## 5. Voz y Visual -- accion con herramientas mock

Ambos agentes llaman a su propia herramienta mock por cada escena del
guion (simulando OmniVoice y Remotion respectivamente).

In [6]:
from agents import visual_agent, voice_agent

voz_resultado = ejecutar(voice_agent.generar_voz(state))
visual_resultado = ejecutar(visual_agent.generar_visual(state, kit="noche"))
voz_resultado, visual_resultado


({'voz': {'resultados': [{'escena': 'E1_gancho',
     'duracion_estimada_s': 4.78,
     'archivo_simulado': 'public/demo/audio_mock/E1_gancho.wav'},
    {'escena': 'E2_producto',
     'duracion_estimada_s': 3.48,
     'archivo_simulado': 'public/demo/audio_mock/E2_producto.wav'},
    {'escena': 'E3_beneficio_1',
     'duracion_estimada_s': 3.04,
     'archivo_simulado': 'public/demo/audio_mock/E3_beneficio_1.wav'},
    {'escena': 'E4_beneficio_2',
     'duracion_estimada_s': 6.09,
     'archivo_simulado': 'public/demo/audio_mock/E4_beneficio_2.wav'},
    {'escena': 'E5_publico',
     'duracion_estimada_s': 3.48,
     'archivo_simulado': 'public/demo/audio_mock/E5_publico.wav'},
    {'escena': 'E6_cierre_cta',
     'duracion_estimada_s': 2.61,
     'archivo_simulado': 'public/demo/audio_mock/E6_cierre_cta.wav'}]},
  'uso': {'input_tokens': 4,
   'cache_creation_input_tokens': 13347,
   'cache_read_input_tokens': 8469,
   'output_tokens': 1608,
   'output_tokens_details': {'thinking_toke

## 6. QA -- chequeo deterministico (no es una llamada a un LLM)

Compara la duracion total estimada (sumando lo que devolvio la herramienta
de voz, no una suposicion del LLM) contra la duracion objetivo, y verifica
que las escenas del guion y de la voz coincidan.

In [7]:
from agents import qa_agent

reporte_qa = qa_agent.verificar(state, duracion_objetivo_s=brief["duracion_objetivo_s"])
reporte_qa


{'aprobado': False,
 'duracion_total_estimada_s': 23.48,
 'desviacion': 0.174,
 'problemas': ['Duracion total estimada (23.48s) se desvia +17% del objetivo (20s), fuera de la tolerancia de ±15%.']}

## 7. Replanificacion condicional

Si el QA no aprueba, el Orquestador decide como corregir y se repite el
ciclo Guion -> Voz -> QA una sola vez (limite para que la demo termine en
un numero acotado de llamadas).

In [8]:
llamadas = [plan_resultado, inv_resultado, guion_resultado, voz_resultado, visual_resultado]

if not reporte_qa["aprobado"] and state.intentos_replanificacion < 1:
    replan_resultado = ejecutar(orchestrator.replanificar(
        state, observaciones="; ".join(reporte_qa["problemas"])
    ))
    llamadas.append(replan_resultado)
    instruccion = replan_resultado["decision"].get("instruccion", "")

    guion_resultado = ejecutar(scriptwriter_agent.escribir_guion(
        state, duracion_objetivo_s=brief["duracion_objetivo_s"], observaciones=instruccion
    ))
    llamadas.append(guion_resultado)

    voz_resultado = ejecutar(voice_agent.generar_voz(state))
    llamadas.append(voz_resultado)

    reporte_qa = qa_agent.verificar(state, duracion_objetivo_s=brief["duracion_objetivo_s"])
    print("QA tras replanificacion:", reporte_qa)
else:
    msg = "QA aprobado en el primer intento." if reporte_qa["aprobado"] else "Se agoto el intento de replanificacion."
    print(msg)


QA tras replanificacion: {'aprobado': True, 'duracion_total_estimada_s': 19.56, 'desviacion': -0.022, 'problemas': []}


## 8. Memoria final (`ReelState`) y bitacora de acciones

In [9]:
print(state.resumen_json())


{
  "brief": {
    "tema": "ahorro_voluntario",
    "cliente_demo": "Cooperativa Demo (dato ficticio para esta demo)",
    "duracion_objetivo_s": 20
  },
  "plan": [
    "investigacion",
    "guion",
    "voz_y_visual",
    "qa"
  ],
  "dossier": {
    "tema": "ahorro_voluntario",
    "datos": {
      "producto": "Ahorro voluntario programado",
      "beneficio_clave": "los aportes quedan disponibles en cualquier momento, sin plazo fijo",
      "publico": "asociados activos de la cooperativa",
      "dato_ejemplo": "tasa de referencia DTF + 2 puntos"
    },
    "advertencias": [
      "El dato de ejemplo (DTF + 2 puntos) es una cifra de catálogo demo según la fuente; debe confirmarse con el cliente antes de publicarse.",
      "La fuente no incluye tasa efectiva verificada, condiciones de retiro, montos mínimos ni plazos; no se reportan."
    ]
  },
  "guion": [
    {
      "id": "E1_gancho",
      "duracion_s": 3,
      "texto_pantalla": "¿Ahorrar sin plazo fijo?",
      "voz": "¿Y si

## 9. Costo real medido

No es un estimado: es la suma de lo que el SDK reporto en cada llamada
(ver `ResultMessage.usage` / `total_cost_usd` en `agents/_sdk_helpers.py`).

In [10]:
total_input = sum((l.get("uso") or {}).get("input_tokens", 0) for l in llamadas)
total_output = sum((l.get("uso") or {}).get("output_tokens", 0) for l in llamadas)
total_costo = sum(l.get("costo_usd", 0) or 0 for l in llamadas)

print(f"Llamadas al modelo: {len(llamadas)}")
print(f"Tokens de entrada: {total_input}")
print(f"Tokens de salida: {total_output}")
print(f"Costo medido por el SDK: ${total_costo:.4f} USD")


Llamadas al modelo: 8
Tokens de entrada: 24
Tokens de salida: 12387
Costo medido por el SDK: $0.1225 USD
